## RAG Pipeline-Data Ingestion to Vector Db Pipeline 

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
import langchain
print(langchain.__version__)

1.3.14


In [11]:
### Read all pds inside the data folder as well as all content and load it 


def process_all_pdf(pdf_directory):
    all_document=[]
    pdf_dir=Path(pdf_directory)

    ## Find All PDF Files Recursively


    pdf_files=list(pdf_dir.glob("**/*pdf"))

    print(f"found{len(pdf_files)} pdf fies to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            document=loader.load()

            #add Source information to metadat

            for doc in document:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]="pdf"

            all_document.extend(document)
            print(f"loaded Succesfully {len(document)} pages")
        except Exception as e:
            print(f"error {e}")
    
    print(f"total document loaded : {len(all_document)}")

    return all_document


all_pdf_documents=process_all_pdf("../data")


    




found4 pdf fies to process

Processing : pdf
error File path ..\data\pdf is not a valid file or url

Processing : Automatic_Infotech_Interview_Prep_Pruthviraj.pdf
loaded Succesfully 18 pages

Processing : Sentinel_AI_Orvexteam_InnoVent2026_1.pdf
loaded Succesfully 16 pages

Processing : Technical_Interview_Questions_Pruthviraj.pdf
loaded Succesfully 21 pages
total document loaded : 55


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf\\Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'file_type': 'pdf'}, page_content='Pruthviraj Thorbole — Automatic Infotech Interview Prep   |   Page 1 of 18 \nINTERVIEW PREPARATION GUIDE \nAI Lab Intern + PPO Hiring — Automatic Infotech, Pune \nPrepared for: Pruthviraj Thorbole \nB.Tech CSE, ADCET Ashta  |  CPI 8.81 \nCommunication Round  →  Technical Interview 1  →  Technical Interview 2  →  Project Assignment  →  Final Offer'),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf

In [13]:
def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """ split documents into smaller chunks for batter RAG Performance"""

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]

    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")


    if split_docs:
        print(f"example chunk: ")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"metadat: {split_docs[0].metadata}")

    return split_docs

In [14]:
chunks=split_documents(all_pdf_documents)

chunks

split 55 documents into 163 chunks
example chunk: 
content: Pruthviraj Thorbole — Automatic Infotech Interview Prep   |   Page 1 of 18 
INTERVIEW PREPARATION GUIDE 
AI Lab Intern + PPO Hiring — Automatic Infotech, Pune 
Prepared for: Pruthviraj Thorbole 
B.Tec...
metadat: {'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf\\Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf\\Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'file_type': 'pdf'}, page_content='Pruthviraj Thorbole — Automatic Infotech Interview Prep   |   Page 1 of 18 \nINTERVIEW PREPARATION GUIDE \nAI Lab Intern + PPO Hiring — Automatic Infotech, Pune \nPrepared for: Pruthviraj Thorbole \nB.Tech CSE, ADCET Ashta  |  CPI 8.81 \nCommunication Round  →  Technical Interview 1  →  Technical Interview 2  →  Project Assignment  →  Final Offer'),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf

## Embadding And Vector DB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: "f"{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode( texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12469.00it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Pruthviraj Thorbole\AppData\Local\Temp\ipykernel_22208\3809748178.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: "f"{self.model.get_sentence_embedding_dimension()}")


## Vectore Store


In [17]:

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__( self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store", ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={ "description": "PDF document embeddings for RAG" },
            )

            print(  f"Vector store initialized. Collection: {self.collection_name}" )
            print(  f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents( self, documents: List[Any], embeddings: np.ndarray,):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(    "Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate( zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                  ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )

            print(f"Successfully added {len(documents)} documents to vector store" )
            print(    f"Total documents in collection: {self.collection.count()}" )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 163


In [18]:
chunks

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf\\Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Automatic_Infotech_Interview_Prep_Pruthviraj.pdf', 'file_type': 'pdf'}, page_content='Pruthviraj Thorbole — Automatic Infotech Interview Prep   |   Page 1 of 18 \nINTERVIEW PREPARATION GUIDE \nAI Lab Intern + PPO Hiring — Automatic Infotech, Pune \nPrepared for: Pruthviraj Thorbole \nB.Tech CSE, ADCET Ashta  |  CPI 8.81 \nCommunication Round  →  Technical Interview 1  →  Technical Interview 2  →  Project Assignment  →  Final Offer'),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-23T02:36:07+05:30', 'author': 'Un-named', 'moddate': '2026-07-23T02:36:07+05:30', 'source': '..\\data\\pdf

In [19]:
### convert the text into chunks
texts=[doc.page_content for doc in chunks]

### generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)


### Store in the vector database


vectorstore.add_documents(chunks,embeddings)

NameError: name 'embedding_manager' is not defined

## Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

rag_retriever

In [ ]:
rag_retriever.retrieve("what is automatic infotech you know")